# Hybrid ranker: baselines + ablation (Kaggle GPU)

Answers the ICMLDE reviewers: every signal of the hybrid score (semantic, EMA, LightGCN, KG) alone,
the hybrid with the paper's weights, and **all 16 on/off combinations of the four signals, each with its
own validation-tuned weights** (the empty combination is popularity, the full one is the tuned hybrid) --
all on one chronological split with Recall / NDCG / MAP / MRR / HitRate / Precision @10 and @20.
The report JSON keeps the earlier `results` block (singles, hand-set and tuned hybrid, fixed-weight
leave-one-out) and adds a `factorial` block with the 16 combinations, their weights and Wilcoxon p-values.

**Run it as a committed run:** Settings -> Accelerator `GPU T4 x2`, Internet On, then
**Save Version -> Save & Run All (Commit)**. The code is pulled from `origin/main`, so push first.

All three domains run in one go (roughly 1-1.5 h each, mostly LightGCN training). Outputs land in
`/kaggle/working/reports/`: one `<domain>_ablation.json` + `.md` per domain. Download them into the
repo's `reports/benchmarks/`.

In [ ]:
# ---------------- CONFIG ----------------
REPO_URL = "https://github.com/sanjay-pokee/Real-Time-Adaptive-Cross-Media-Recommendation-System.git"

# Same graphs as reports/domain_benchmark_table.md, so the numbers line up.
DOMAINS = {
    "Movies_and_TV":             {"max_users": 50000, "core": 10},
    "Health_and_Household":      {"max_users": 50000, "core": 10},
    "Industrial_and_Scientific": {"max_users": 0,     "core": 5},
}
EPOCHS     = 60
BATCH_SIZE = 4096
VAL_USERS  = 10000   # users used for the weight grid search
EVAL_BATCH = 512     # lower to 256 on out-of-memory
REPORT_DIR = "/kaggle/working/reports"
# ----------------------------------------

In [ ]:
import os, subprocess, sys
WORK = "/kaggle/working"
REPO_DIR = os.path.join(WORK, "repo")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub", "pandas", "pyarrow", "sentence-transformers", "scipy"], check=True)

def run_step(*args):
    # -u: unbuffered, otherwise a long step looks like a hang.
    subprocess.run([sys.executable, "-u", "-m", *args], check=True)

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import glob, time, traceback
os.makedirs(REPORT_DIR, exist_ok=True)
failed = []
for category, cfg in DOMAINS.items():
    started = time.time()
    slug = category.lower()
    print("=" * 70)
    print(category)
    print("=" * 70, flush=True)
    # One domain failing (OOM, download hiccup) must not cost the other two.
    try:
        run_step("scripts.fetch_amazon_dataset", "--category", category)
        raw = glob.glob(f"datasets/amazon/**/{category}.csv", recursive=True)[0]

        prepared = f"{WORK}/prepared_{slug}_{cfg['max_users']}.parquet"
        if not os.path.exists(prepared):
            run_step("models.graph.datasets", raw,
                     "--user-core", str(cfg["core"]), "--item-core", str(cfg["core"]),
                     "--max-users", str(cfg["max_users"]), "--out", prepared)

        meta = f"datasets/amazon/meta/{category}_benchmark.csv"
        if not os.path.exists(meta):
            run_step("scripts.fetch_amazon_metadata", "--category", category,
                     "--asins-from", prepared, "--out-name", f"{category}_benchmark")

        run_step("models.graph.evaluate_hybrid_ablation",
                 "--dataset", prepared, "--metadata", meta,
                 "--user-core", str(cfg["core"]), "--item-core", str(cfg["core"]),
                 "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
                 "--val-users", str(VAL_USERS), "--eval-batch", str(EVAL_BATCH),
                 "--graph-cache", f"{WORK}/graph_cache_{slug}.npz",
                 "--report", f"{REPORT_DIR}/{slug}_ablation.json")
        print(f"{category} done in {(time.time() - started) / 60:.1f} min", flush=True)
    except Exception:
        traceback.print_exc()
        failed.append(category)
        print(f"{category} FAILED after {(time.time() - started) / 60:.1f} min", flush=True)

print("failed domains:", failed or "none")

In [ ]:
for path in sorted(glob.glob(f"{REPORT_DIR}/*_ablation.md")):
    print(open(path).read())